# 03 - Xử lý dữ liệu thiếu và trùng lặp

## Mục tiêu

- Đọc dữ liệu đã tích hợp từ bước 1.
- Kiểm tra dữ liệu thiếu.
- Xử lý dữ liệu thiếu.
- Kiểm tra và loại bỏ dữ liệu trùng lặp.
- Kiểm tra dữ liệu không hợp lệ.
- Xuất dữ liệu sau khi làm sạch.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# đọc dữ liệu đã tích hợp từ bước 1
df = pd.read_csv('../data/processed/step1_merged.csv')
print("Kích thước dữ liệu:", df.shape)

Kích thước dữ liệu: (112650, 26)


In [3]:
# xem 5 dòng đầu tiên
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,barreiras,BA,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,...,vianopolis,GO,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,...,sao goncalo do amarante,RN,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,...,santo andre,SP,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0


In [4]:
# kiểm tra dữ liệu thiếu
missing = df.isna().sum()
missing_percent = (df.isna().mean() * 100)

missing_summary = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': missing_percent
})
print(missing_summary[missing_summary['missing_count'] > 0]
      .sort_values('missing_percent', ascending=False))

                               missing_count  missing_percent
order_delivered_customer_date           2454         2.178429
product_name_lenght                     1603         1.422992
product_category_name                   1603         1.422992
product_description_lenght              1603         1.422992
product_photos_qty                      1603         1.422992
order_delivered_carrier_date            1194         1.059920
product_weight_g                          18         0.015979
product_height_cm                         18         0.015979
product_length_cm                         18         0.015979
product_width_cm                          18         0.015979
order_approved_at                         15         0.013316


In [11]:
# 1️⃣ NHÓM 1: Kích thước sản phẩm (< 0.05% thiếu) → DROP DÒNG
df = df.dropna(subset=['product_weight_g', 'product_height_cm', 'product_length_cm', 'product_width_cm'])

# 2️⃣ NHÓM 2: Thông tin sản phẩm (~1.4% thiếu) → FILL VALUE
df['product_category_name'] = df['product_category_name'].fillna('Unknown')
df['product_name_lenght'] = df['product_name_lenght'].fillna(df['product_name_lenght'].median())
df['product_description_lenght'] = df['product_description_lenght'].fillna(df['product_description_lenght'].median())
df['product_photos_qty'] = df['product_photos_qty'].fillna(df['product_photos_qty'].median())

# 3️⃣ NHÓM 3: Ngày tháng đơn hàng → GIỮ NGUYÊN MISSING
# Thiếu = đơn hàng chưa duyệt/giao hoặc bị hủy (có ý nghĩa kinh doanh)
print("Missing giữ nguyên:")
print(f"  - order_approved_at: {df['order_approved_at'].isna().sum()} dòng")
print(f"  - order_delivered_carrier_date: {df['order_delivered_carrier_date'].isna().sum()} dòng")
print(f"  - order_delivered_customer_date: {df['order_delivered_customer_date'].isna().sum()} dòng")

Missing giữ nguyên:
  - order_approved_at: 15 dòng
  - order_delivered_carrier_date: 1194 dòng
  - order_delivered_customer_date: 2454 dòng


In [6]:
# kiểm tra lại dữ liệu thiếu sau xử lý
print(df.isna().sum()[df.isna().sum() > 0])

order_approved_at                  15
order_delivered_carrier_date     1194
order_delivered_customer_date    2454
dtype: int64


## Xử lý dữ liệu trùng lặp

In [7]:
# kiểm tra dữ liệu trùng lặp
dup_count = df.duplicated().sum()
print("Số dòng trùng lặp:", dup_count)

df = df.drop_duplicates()
print("Sau khi loại bỏ:", df.duplicated().sum())

Số dòng trùng lặp: 0
Sau khi loại bỏ: 0


In [8]:
# kiểm tra dữ liệu không hợp lệ
print("Price < 0:", (df['price'] < 0).sum())
print("Freight < 0:", (df['freight_value'] < 0).sum())
print("order_item_id <= 0:", (df['order_item_id'] <= 0).sum())

df = df[df['price'] >= 0]
df = df[df['freight_value'] >= 0]
df = df[df['order_item_id'] > 0]

print("Kích thước sau xử lý:", df.shape)

Price < 0: 0
Freight < 0: 0
order_item_id <= 0: 0
Kích thước sau xử lý: (112632, 26)


#  KHÁM PHÁ OUTLIERS

In [12]:
# Phương pháp 1: IQR (Interquartile Range)
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Price - Ngưỡng dưới: {lower_bound:.2f}, Ngưỡng trên: {upper_bound:.2f}")
outliers = df[(df['price'] < lower_bound) | (df['price'] > upper_bound)]
print(f"Số ngoại lai price: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

# Tương tự với freight_value
Q1_f = df['freight_value'].quantile(0.25)
Q3_f = df['freight_value'].quantile(0.75)
IQR_f = Q3_f - Q1_f
lower_bound_f = Q1_f - 1.5 * IQR_f
upper_bound_f = Q3_f + 1.5 * IQR_f

print(f"\nFreight - Ngưỡng dưới: {lower_bound_f:.2f}, Ngưỡng trên: {upper_bound_f:.2f}")
outliers_f = df[(df['freight_value'] < lower_bound_f) | (df['freight_value'] > upper_bound_f)]
print(f"Số ngoại lai freight: {len(outliers_f)} ({len(outliers_f)/len(df)*100:.2f}%)")

print("\n⚠️ Lưu ý: Xử lý outliers sẽ thực hiện ở bước 4")

Price - Ngưỡng dưới: -102.60, Ngưỡng trên: 277.40
Số ngoại lai price: 8426 (7.48%)

Freight - Ngưỡng dưới: 0.98, Ngưỡng trên: 33.25
Số ngoại lai freight: 12134 (10.77%)

⚠️ Lưu ý: Xử lý outliers sẽ thực hiện ở bước 4


In [13]:
# lưu kết quả vào file csv
df.to_csv('../data/processed/step3_cleaned.csv', index=False)
print("Đã lưu step3_cleaned.csv, kích thước:", df.shape)

Đã lưu step3_cleaned.csv, kích thước: (112632, 26)


## Kết luận

Bước 3 đã xử lý dữ liệu thành công:

**Xử lý Missing Values** → Fill/Drop theo lý do kinh doanh  
**Loại bỏ Duplicates** → Xóa toàn bộ dòng trùng lặp  
**Xóa Invalid Data** → Giá âm, freight âm, order_item_id <= 0  

**Kết quả:** Dataset sạch, sẵn sàng cho bước 4 (Xử lý Outliers)